In [1]:
from catalyst.debug.compiler_functions import get_compilation_stage
import pennylane as qml
from catalyst.third_party.oqd import OQDDevice

import os
import shutil
import pathlib
import numpy as np

########################################################################################

for f in os.listdir():
    if f.startswith("oqd_circuit") and os.path.isdir(f):
        shutil.rmtree(pathlib.Path(f))

compile_results = pathlib.Path("oqd_circuit")
openapl_file_name = "oqd_circuit.openapl.json"

oqd_dev = OQDDevice(
    backend="default",
    wires=2,
    openapl_file_name=(compile_results / openapl_file_name).as_posix(),
)
oqd_dev.openapl_file_name


toml_files = {
    "device-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/device.toml",
    "qubit-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml",
    "gate-to-pulse-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/gate.toml",
}

toml_files = " ".join([f"{k}={v}" for k, v in toml_files.items()])


OQD_PIPELINES = [
    (
        "DeviceAgnosticPipeline",
        [
            "quantum-compilation-stage",
            "hlo-lowering-stage",
            "gradient-lowering-stage",
            "bufferization-stage",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "func.func(ions-decomposition)",
            # "func.func(merge-rotations)",
            # "func.func(prune-zero-rotations)",
        ],
    ),
    (
        "IonDialectLoweringStage",
        [
            f"func.func(gates-to-pulses{{{toml_files}}})",
        ],
    ),
    ("IonToLLVMDialectConversion", ["convert-ion-to-llvm"]),
    ("MLIRToLLVMDialectConversion", ["llvm-dialect-lowering-stage"]),
]


@qml.qjit(keep_intermediate=True, pipelines=OQD_PIPELINES, verbose=True)
@qml.set_shots(1)
@qml.qnode(oqd_dev)
def oqd_circuit():
    qml.H(wires=[0])
    qml.CNOT(wires=[0, 1])

    qml.measure(wires=[0])

    return qml.counts(wires=[0])


print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(oqd_circuit, stage="IonDialectLoweringStage"))


[LIB] Running compiler driver in /home/user/oqd-catalyst/scripts/oqd_circuit
[SYSTEM] /home/user/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/oqd-catalyst/scripts/oqd_circuit/oqd_circuit.ll --module-name oqd_circuit --workspace /home/user/oqd-catalyst/scripts/oqd_circuit -verify-each=false --catalyst-pipeline DeviceAgnosticPipeline(quantum-compilation-stage;hlo-lowering-stage;gradient-lowering-stage;bufferization-stage),IonDecompositionStage(func.func(ions-decomposition)),IonDialectLoweringStage(func.func(gates-to-pulses{device-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/device.toml qubit-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml gate-to-pulse-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/gate.toml})),IonToLLVMDialectConversion(convert-ion-to-llvm),MLIRToLLVMDialectConversion(llvm-dialect-lowering-stage), --keep-intermediate --verbose /home/user/oqd-catalyst/scripts/oqd_circuit/tmpcl60h_kb.mlir


In [2]:
import json

oqd_circuit()

print(json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2))

{
  "class_": "AtomicCircuit",
  "protocol": {
    "class_": "SequentialProtocol",
    "sequence": [
      {
        "class_": "ParallelProtocol",
        "sequence": [
          {
            "beam": {
              "class_": "Beam",
              "detuning": {
                "class_": "MathNum",
                "value": 208570336271826.38
              },
              "phase": {
                "class_": "MathNum",
                "value": 0.0
              },
              "polarization": [
                1,
                0,
                0
              ],
              "rabi": {
                "class_": "MathNum",
                "value": 6283185307.179586
              },
              "target": 0,
              "transition": {
                "class_": "Transition",
                "einsteinA": 41050903.119868636,
                "label": "downstate->estate",
                "level1": {
                  "class_": "Level",
                  "energy": 0.0,
               

In [3]:
from oqd_core.interface.atomic import AtomicCircuit
from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
from oqd_compiler_infrastructure import Chain, Post
from oqd_bare_metal.compiler.codegen import AtomicToTestbenchV2
from oqd_bare_metal.compiler.optim import (
    SpectrumCoreRemapping,
    SpectrumPrune,
    SpectrumUnwrapResets,
)
import ast_comments as ast


circuit = AtomicCircuit.model_validate_json(
    json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
)


compiler = Chain(
    canonicalize_atomic_circuit_factory(),
    Post(AtomicToTestbenchV2(device_params="./calibration_data/testbench_params.toml")),
)
optimization_pass = Chain(
    Post(SpectrumCoreRemapping(device="raman_awg")),
    Post(SpectrumUnwrapResets()),
    Post(SpectrumPrune()),
)

unopt_artiq_experiment = compiler(circuit)
artiq_experiment = optimization_pass(unopt_artiq_experiment)


with open(compile_results / "oqd_circuit.artiq.py", "w") as f:
    f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

In [4]:
# from oqd_core.interface.atomic import AtomicCircuit
# from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
# from oqd_compiler_infrastructure import Chain, Post
# from oqd_bare_metal.compiler.codegen import AtomicToBloodstoneV1
# from oqd_bare_metal.compiler.optim import (
#     SpectrumCoreRemapping,
#     SpectrumPrune,
#     SpectrumUnwrapResets,
# )
# import ast_comments as ast


# circuit = AtomicCircuit.model_validate_json(
#     json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
# )


# compiler = Chain(
#     canonicalize_atomic_circuit_factory(),
#     Post(
#         AtomicToBloodstoneV1(device_params="./calibration_data/bloodstone_params.toml")
#     ),
# )
# optimization_pass = Chain(
#     Post(SpectrumCoreRemapping(device="raman_awg")),
#     Post(SpectrumUnwrapResets()),
#     Post(SpectrumPrune()),
# )

# unopt_artiq_experiment = compiler(circuit)
# artiq_experiment = optimization_pass(unopt_artiq_experiment)


# with open(compile_results / "oqd_circuit.artiq.py", "w") as f:
#     f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))